In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from typing import Optional, Literal

Load original pick data CSV into dataframe.

In [2]:
filename = r"C:\Users\LindseyBuss\Documents\OBETA_Project\003 pick_data.csv"
column_names = ['product_id', 'warehouse_section', 'origin', 'order_number', 'position_in_order', 'pick_volume', 'quantity_unit', 'date']
data_types = {
    'product_id': 'string', 
   'warehouse_section': 'category',
   'origin': 'category',
   'order_number': 'string',
   'position_in_order':'int64',
   'pick_volume': 'int64',
   'quantity_unit': 'string',
   'date': 'string'
}
original_pick_data_df = pd.read_csv(filename, names=column_names, header=None, dtype=data_types, parse_dates=['date'])
print(original_pick_data_df.head())

  product_id warehouse_section origin order_number  position_in_order  \
0     000002               SHL     48     07055448                  1   
1     000002               SHL     48     07055448                  1   
2     000002               SHL     48     07055448                  1   
3     000002               SHL     48     07055448                  1   
4     000002               SHL     48     07055448                  1   

   pick_volume quantity_unit                date  
0           29            St 2017-06-30 11:15:24  
1           30            St 2017-06-30 11:22:35  
2           30            St 2017-06-30 12:04:50  
3           20            St 2017-06-30 12:04:51  
4           30            St 2017-06-30 12:05:02  


In [3]:
# Add unique pick IDs.
original_pick_data_df['pick_id'] = range(1, len(original_pick_data_df) + 1)

# Extract year of order to create new unique order IDs.
original_pick_data_df['year_of_order'] = original_pick_data_df['date'].dt.year.astype(str)

original_pick_data_df['updated_order_number'] = original_pick_data_df[['order_number', 'year_of_order']].astype(str).agg('-'.join, axis=1)
print(original_pick_data_df.head())

  product_id warehouse_section origin order_number  position_in_order  \
0     000002               SHL     48     07055448                  1   
1     000002               SHL     48     07055448                  1   
2     000002               SHL     48     07055448                  1   
3     000002               SHL     48     07055448                  1   
4     000002               SHL     48     07055448                  1   

   pick_volume quantity_unit                date  pick_id year_of_order  \
0           29            St 2017-06-30 11:15:24        1          2017   
1           30            St 2017-06-30 11:22:35        2          2017   
2           30            St 2017-06-30 12:04:50        3          2017   
3           20            St 2017-06-30 12:04:51        4          2017   
4           30            St 2017-06-30 12:05:02        5          2017   

  updated_order_number  
0        07055448-2017  
1        07055448-2017  
2        07055448-2017  
3        0

In [4]:
# Drop null values, duplicates, original order number and picks with a volume of zero.
original_pick_data_df = original_pick_data_df.drop(columns='order_number')
original_pick_data_df = original_pick_data_df.dropna()
original_pick_data_df = original_pick_data_df.drop_duplicates()
original_pick_data_df = original_pick_data_df[original_pick_data_df['pick_volume'] != 0]

#cleaned_pick_data = original_pick_data_df
# cleaned_pick_data.to_parquet()

In [5]:

def flag_outliers_by_unit(
    df: pd.DataFrame,
    value_col: str,
    *,
    group_col: str = "quantity_unit",
    method: Literal["zscore", "modified"] = "zscore",
    threshold: Optional[float] = None,
    ddof: int = 0,
    min_group_size: int = 3,
    separate_sides: bool = False,
    flag_col: Optional[str] = None,
) -> pd.DataFrame:
    """
    Flag outliers in `value_col` *per group* defined by `group_col` (e.g., quantity_unit).

    Parameters
    ----------
    df : pd.DataFrame
        Input DataFrame.
    value_col : str
        Numeric column to analyze for outliers.
    group_col : str
        Grouping column (default: 'quantity_unit').
    method : {'zscore', 'modified'}
        'zscore' = mean/std; 'modified' = median/MAD (robust).
    threshold : float, optional
        Threshold for the chosen method. Defaults: 3.0 for zscore, 3.5 for modified.
    ddof : int
        ddof for std in zscore method.
    min_group_size : int
        Minimum group size to compute outliers. Smaller groups are marked as non-outliers.
    separate_sides : bool
        If True, adds two columns: lower/upper outliers (instead of a single boolean).
    flag_col : str, optional
        Name of the output flag column when separate_sides=False.
        Defaults to f'{value_col}_is_outlier_by_{group_col}'.

    Returns
    -------
    pd.DataFrame
        Copy of df with added outlier flag column(s), computed per group.

    Notes
    -----
    - NaNs in value_col are ignored in stats; rows with NaN are flagged as non-outliers.
    - Constant groups (std=0 or MAD=0) produce no outliers.
    """

    if threshold is None:
        threshold = 3.0 if method == "zscore" else 3.5
    if flag_col is None and not separate_sides:
        flag_col = f"{value_col}_is_outlier_by_{group_col}"

    out = df.copy()

    # Ensure numeric dtype (will convert errors to NaN)
    x = pd.to_numeric(out[value_col], errors="coerce")

    def compute_group_flags(g: pd.Series) -> pd.DataFrame:
        # g is the value_col Series for one group
        # Build a DataFrame with aligned index for returning multiple columns
        res = pd.DataFrame(index=g.index)

        # Respect minimum group size
        valid = g.dropna()
        if valid.size < min_group_size:
            if separate_sides:
                res["is_lower_outlier"] = False
                res["is_upper_outlier"] = False
            else:
                res[flag_col] = False
            return res

        if method == "zscore":
            mean = valid.mean()
            std = valid.std(ddof=ddof)
            if std == 0 or np.isnan(std):
                z = pd.Series(0.0, index=g.index)
            else:
                z = (g - mean) / std
        elif method == "modified":
            med = valid.median()
            mad = (valid - med).abs().median()
            if mad == 0 or np.isnan(mad):
                z = pd.Series(0.0, index=g.index)
            else:
                z = 0.6745 * (g - med) / mad
        else:
            raise ValueError("method must be 'zscore' or 'modified'")

        # Build masks; treat NaNs as non-outliers
        if separate_sides:
            res["is_lower_outlier"] = z.lt(-threshold).fillna(False)
            res["is_upper_outlier"] = z.gt(threshold).fillna(False)
        else:
            res[flag_col] = z.abs().gt(threshold).fillna(False)

        return res

    flags = x.groupby(out[group_col], dropna=False).apply(compute_group_flags)
    # groupby.apply returns a nested index; drop the group level
    flags.index = flags.index.get_level_values(-1)

    # Merge flags back
    for col in flags.columns:
        out[col] = flags[col]

    return out


  


In [6]:

pick_data_with_outliers = flag_outliers_by_unit(
    original_pick_data_df,
    value_col="pick_volume",
    group_col="quantity_unit",
    method="zscore",
    threshold=3.0,
)
pick_data_with_outliers.head()


,product_id,warehouse_section,origin,position_in_order,pick_volume,quantity_unit,date,pick_id,year_of_order,updated_order_number,pick_volume_is_outlier_by_quantity_unit
0,000002,SHL,48,1,29,St,2017-06-30 11:15:24,1,2017,07055448-2017,False
1,000002,SHL,48,1,30,St,2017-06-30 11:22:35,2,2017,07055448-2017,False
2,000002,SHL,48,1,30,St,2017-06-30 12:04:50,3,2017,07055448-2017,False
3,000002,SHL,48,1,20,St,2017-06-30 12:04:51,4,2017,07055448-2017,False
4,000002,SHL,48,1,30,St,2017-06-30 12:05:02,5,2017,07055448-2017,False


In [7]:
cleaned_pick_data = pick_data_with_outliers
cleaned_pick_data.to_csv('cleaned_pick_data.csv')

Subset data and prepare order summary information. Subset by year to ensure all picks from the same order are processed in the same subset.

In [8]:
years = cleaned_pick_data['year_of_order'].unique()


In [9]:
def summarize_year(df):
    order_summary_data = df.groupby('updated_order_number').agg(
        origin = ('origin', 'first'),
        num_picks = ('pick_volume', 'sum'),
        num_products = ('product_id', 'nunique'),
        num_sections = ('warehouse_section', 'nunique'),
        num_positions = ('position_in_order', 'nunique'),
        time_of_first_pick = ('date', 'min'),
        time_of_last_pick = ('date', 'max'))

    order_summary_data['time_to_fulfil'] = order_summary_data['time_of_last_pick'] - order_summary_data['time_of_first_pick']
    order_summary_data['date'] = order_summary_data['time_of_first_pick'].dt.date

    return order_summary_data

In [10]:
order_summaries_by_year = []

for year in years:
    df_subset = cleaned_pick_data[cleaned_pick_data['year_of_order'] == year]
    summarized_year = summarize_year(df_subset)
    order_summaries_by_year.append(summarized_year)

final_order_data = pd.concat(order_summaries_by_year)
print(final_order_data.head())

# final_order_data.to_parquet()



                     origin  num_picks  num_products  num_sections  \
updated_order_number                                                 
04154598-2017            48          1             1             1   
04155767-2017            48          2             1             1   
04155790-2017            48        282             7             2   
04155811-2017            48         14            10             2   
04155813-2017            48         69             5             2   

                      num_positions  time_of_first_pick   time_of_last_pick  \
updated_order_number                                                          
04154598-2017                     1 2017-01-02 14:22:45 2017-01-02 14:22:45   
04155767-2017                     1 2017-01-05 19:03:09 2017-01-05 19:03:09   
04155790-2017                     7 2017-01-02 12:20:57 2017-01-02 12:25:23   
04155811-2017                    10 2017-01-02 12:19:34 2017-01-02 12:34:51   
04155813-2017                     5